In [ ]:
# Consolidate everything into a single, clean installation
!pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt
!pip install -U "jax[cuda12]"
!pip install tensorflow==2.19.0 tensorflow-text==2.19.0
!pip install t5 seqio
!pip install nest-asyncio

# Clone T5X if not present
import os
if not os.path.exists("/kaggle/working/t5x"):
    !git clone https://github.com/google-research/t5x /kaggle/working/t5x
%cd /kaggle/working/t5x
!pip install -q -e .


# Patch t5x partitioning.py for GPU compatibility
import fileinput
part_file = "/kaggle/working/t5x/t5x/partitioning.py"
if os.path.exists(part_file):
    for line in fileinput.input(part_file, inplace=True):
        if "return x + 1, y + 1, z + 1, last_device.core_on_chip + 1" in line:
            print("    return x + 1, y + 1, z + 1, getattr(last_device, \"core_on_chip\", 0) + 1")
        elif "return (*device.coords, device.core_on_chip)" in line:
            print("    return (*device.coords, getattr(device, \"core_on_chip\", 0))")
        else:
            print(line, end="")


In [ ]:
import sys
import os
sys.path.insert(0, '/kaggle/working/t5x')

import jax
import seqio
import t5x
print(f"✅ JAX Backend: {jax.default_backend()}")
print("🚀 T5X is fully loaded!")


In [ ]:
# Create a directory for your checkpoint
!mkdir -p /kaggle/working/checkpoints/t5_base/

# Copy the checkpoint from Google's public bucket
!gsutil -m cp -r gs://t5-data/pretrained_models/t5x/t5_base/checkpoint_999900 /kaggle/working/checkpoints/t5_base/


In [ ]:
try:
    import t5.data.tasks
    import seqio
    print("✅ T5 Data Tasks and SeqIO are ready.")
    print(f"✅ Available Tasks (sample): {list(seqio.TaskRegistry.names())[:5]}")
except Exception as e:
    import traceback; print(f"❌ Still an issue: {traceback.format_exc()}")


In [ ]:
import seqio
import t5.data.tasks
wmt_tasks = [name for name in seqio.TaskRegistry.names() if 'wmt14_en_de' in name]


In [ ]:
import seqio
import t5.data
import t5.evaluation.metrics
import functools

# 1. Clear the registry
if "wmt14_en_de_v003" in seqio.TaskRegistry.names():
    seqio.TaskRegistry.remove("wmt14_en_de_v003")

# 2. Use the exact vocab from your previous model error
model_vocab = seqio.SentencePieceVocabulary(
    "gs://t5-data/vocabs/cc_all.32000.100extra/sentencepiece.model", 
    extra_ids=0
)

# 3. Define the preprocessor with the missing arguments
# We use functools.partial to bake 'en' and 'de' into the function call
translate_preprocessor = functools.partial(
    t5.data.preprocessors.translate, 
    source_language='en', 
    target_language='de'
)

# 4. Re-register
seqio.TaskRegistry.add(
    "wmt14_en_de_v003",
    source=seqio.TfdsDataSource(tfds_name="wmt14_translate/de-en:1.0.0"),
    preprocessors=[
        translate_preprocessor, # Now has the required args!
        seqio.preprocessors.tokenize,
        seqio.preprocessors.append_eos,
    ],
    output_features={
        "inputs": seqio.Feature(vocabulary=model_vocab, add_eos=True),
        "targets": seqio.Feature(vocabulary=model_vocab, add_eos=True)
    },
    metric_fns=[t5.evaluation.metrics.bleu]
)


In [ ]:
import os
import sys

# 1. Force Single-GPU & VRAM behavior
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"


# 1. Force Single-GPU & VRAM behavior

# 2. Patch Optax BEFORE importing T5X
import optax
if not hasattr(optax, 'ConditionallyTransformState'):
    class DummyState: pass
    optax.ConditionallyTransformState = DummyState
    print("🛠️ Optax Patch Applied: Added ConditionallyTransformState")

# 3. Now it is safe to import T5X
import jax
import gin
import __main__
import seqio
import t5.data.tasks
from t5x import eval as t5x_eval_lib

# 4. Patch JAX Config for Shardy (as we did before)
if not hasattr(jax.config, 'jax_use_shardy_partitioner'):
    setattr(jax.config, 'jax_use_shardy_partitioner', False)

# 5. Inject Namespace for Gin
__main__.evaluate = t5x_eval_lib.evaluate
class EvalScriptSpoof:
    evaluate = t5x_eval_lib.evaluate
__main__.eval_script = EvalScriptSpoof

# 6. Paths & Config
T5X_ROOT = "/kaggle/working/t5x"
# Make sure to use the base model config that specializes in EN-DE translation.
MODEL_CONFIG = os.path.join(T5X_ROOT, "t5x/examples/t5/t5_1_0/base.gin")
EVAL_CONFIG = os.path.join(T5X_ROOT, "t5x/configs/runs/eval.gin")

gin.add_config_file_search_path(os.path.join(T5X_ROOT, "t5x/configs"))
gin.add_config_file_search_path(os.path.join(T5X_ROOT, "t5x/examples"))
gin.add_config_file_search_path(os.path.join(T5X_ROOT, "t5x/contrib/gpu"))

# Note we use the t5 base model and checkpoint and config.
gin_bindings = [
    "DROPOUT_RATE = 0.0",
    "eval_script.evaluate.use_orbax=False",
    "CHECKPOINT_PATH='/kaggle/working/checkpoints/t5_base/checkpoint_999900'",
    "EVAL_OUTPUT_DIR='/kaggle/working/eval_results'",
    "MIXTURE_OR_TASK_NAME='wmt14_en_de_v003'",
    "utils.DatasetConfig.batch_size=8", 
    "utils.DatasetConfig.use_cached=False", 
    "infer_checkpoint_step=999900",
    "partitioning.PjitPartitioner.num_partitions = 1"
]

print(f"✅ JAX visible devices: {jax.local_devices()}")

try:
    gin.clear_config()
    gin.parse_config_files_and_bindings([MODEL_CONFIG, EVAL_CONFIG], gin_bindings)
    configured_evaluate = gin.get_configurable(t5x_eval_lib.evaluate)
    
    print("🚀 Starting T5X Evaluation...")
    import nest_asyncio; nest_asyncio.apply()
    configured_evaluate()
    
except Exception as e:
    import traceback; print(f"❌ Still an issue: {traceback.format_exc()}")


In [ ]:
import os
import glob

print('Contents of /kaggle/working/eval_results:')
for f in glob.glob('/kaggle/working/eval_results/**', recursive=True):
    print(f)
